# Graph Learning - Projeto

Este notebook executa o roteiro do projeto usando as funções auxiliares em `graph_generation.py`, `learning.py` e `stats.py`.


## Configuração


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

from graph_generation import (
    erdos_renyi_graph,
    filter_signal,
    normalize_laplacian_norm,
    random_geometric_graph,
    signal_distances,
    upper_triangle_indices,
)
from learning import solve_kalofolias
from stats import graph_metrics, print_table, summarize


M = 100
N = 1000
SIGMA = 0.2
CUTOFF = 0.6
ER_P = 0.03
ALPHA = 1.0
BETAS = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
SUPPORT_RTOL = 1e-4
REPETITIONS = 20
SEED = 20240713


## Geração dos grafos e sinais

Geramos o grafo geométrico aleatório e o grafo de Erdős-Rényi, normalizamos o Laplaciano de cada um para norma espectral igual a 1, geramos o sinal gaussiano branco e aplicamos os filtros de Tikhonov e calor.


In [ ]:
def build_cases(seed):
    seeds = np.random.SeedSequence(seed).spawn(4)

    W_geo, geo_positions = random_geometric_graph(M, SIGMA, CUTOFF, np.random.default_rng(seeds[0]))
    W_er = erdos_renyi_graph(M, ER_P, np.random.default_rng(seeds[1]))

    graphs_data = {
        "geometric": {"W": W_geo, "positions": geo_positions},
        "erdos_renyi": {"W": W_er, "positions": None},
    }

    cases = {}
    for graph_index, (graph_name, graph_data) in enumerate(graphs_data.items()):
        W_true, _ = normalize_laplacian_norm(graph_data["W"])
        X0 = np.random.default_rng(seeds[2 + graph_index]).standard_normal((M, N))

        for filter_name in ["Tikhonov", "Heat"]:
            X = filter_signal(W_true, X0, filter_name)
            cases[(graph_name, filter_name)] = {
                "W_true": W_true,
                "positions": graph_data["positions"],
                "z": signal_distances(X)[upper_triangle_indices(M)],
            }

    return cases


## Aprendizado do grafo

Para cada caso, fazemos uma busca em `beta`. A escolha é feita usando o F1-score; em empate, usamos o menor erro relativo `l2` dos pesos.


In [ ]:
def learn_case(case, keep_graph=False):
    candidates = []
    W_true = case["W_true"]

    for beta in BETAS:
        W_raw, _, info = solve_kalofolias(case["z"], M, alpha=ALPHA, beta=beta)
        row = {"beta": beta, **info}

        if info["converged"]:
            W_hat, _ = normalize_laplacian_norm(W_raw)
            row.update(graph_metrics(W_true, W_hat, SUPPORT_RTOL))
            if keep_graph:
                row["W_hat"] = W_hat

        candidates.append(row)

    valid = [row for row in candidates if row["converged"]]
    return min(valid, key=lambda row: (-row["f1"], row["edge_mean_relative_error"]))


def run_once(seed, keep_graphs=False):
    cases = build_cases(seed)
    rows = []
    for (graph_name, filter_name), case in cases.items():
        best = learn_case(case, keep_graph=keep_graphs)
        rows.append({"graph": graph_name, "filter": filter_name, **best})
    return rows, cases


## Métricas usadas

Usamos métricas correspondentes às quatro classes pedidas no projeto:

- erro absoluto médio nas arestas originais: `edge_abs_l1_mean`;
- erro quadrático médio nas arestas originais: `edge_abs_l2_mean`;
- erro relativo médio nas arestas originais: `edge_mean_relative_error`;
- erro relativo médio dos graus nos vértices com grau original positivo: `degree_mean_relative_error`;
- precisão, sensibilidade e F1 para o padrão de esparsidade.


## Experimento individual


In [ ]:
single, single_cases = run_once(SEED, keep_graphs=True)
print_table(single, [
    "graph", "filter", "beta",
    "edge_abs_l1_mean", "edge_abs_l2_mean", "edge_mean_relative_error",
    "degree_mean_relative_error", "precision", "sensibilidade", "f1",
])


## Grafos original, aprendido e diferença

Para cada grafo aprendido, mostramos o grafo alvo, o grafo aprendido e o grafo de erro absoluto `abs(W - W_hat)` em vermelho.


In [ ]:
def threshold_matrix(W, threshold):
    W_plot = W.copy()
    W_plot[W_plot <= threshold] = 0.0
    return W_plot


def graph_from_matrix(W, threshold):
    return nx.from_numpy_array(threshold_matrix(W, threshold))


def positions_for_graph(graph_name, W_true, positions):
    if positions is not None:
        return {i: positions[i] for i in range(W_true.shape[0])}
    return nx.spring_layout(graph_from_matrix(W_true, 1e-12), seed=SEED, weight="weight")


def draw_weighted_graph(ax, W, positions, title, color, threshold, width_scale):
    G = graph_from_matrix(W, threshold)
    weights = [data["weight"] for _, _, data in G.edges(data=True)]
    widths = [0.4 + 2.6 * weight / width_scale for weight in weights]
    nx.draw_networkx_edges(G, positions, ax=ax, width=widths, edge_color=color, alpha=0.55)
    nx.draw_networkx_nodes(G, positions, ax=ax, node_size=16, node_color="black")
    ax.set_title(title)
    ax.set_axis_off()
    ax.set_aspect("equal")


def draw_difference_graph(ax, W_diff, positions, title):
    G = graph_from_matrix(W_diff, 1e-12)
    weights = [data["weight"] for _, _, data in G.edges(data=True)]
    width_scale = max(W_diff.max(), 1e-12)
    widths = [0.4 + 3.0 * weight / width_scale for weight in weights]
    colors = [(1.0, 0.0, 0.0, 0.15 + 0.85 * weight / width_scale) for weight in weights]
    nx.draw_networkx_edges(G, positions, ax=ax, width=widths, edge_color=colors)
    nx.draw_networkx_nodes(G, positions, ax=ax, node_size=16, node_color="black")
    ax.set_title(title)
    ax.set_axis_off()
    ax.set_aspect("equal")


def plot_graph_comparisons(single, cases):
    fig, axes = plt.subplots(len(single), 3, figsize=(13, 4 * len(single)))

    for row_index, row in enumerate(single):
        key = (row["graph"], row["filter"])
        W_true = cases[key]["W_true"]
        W_hat = row["W_hat"]
        learned_threshold = SUPPORT_RTOL * W_hat.max() if W_hat.max() > 0 else 0.0
        W_true_plot = threshold_matrix(W_true, 1e-12)
        W_hat_plot = threshold_matrix(W_hat, learned_threshold)
        W_diff = abs(W_true_plot - W_hat_plot)
        positions = positions_for_graph(row["graph"], W_true, cases[key]["positions"])
        width_scale = max(W_true_plot.max(), W_hat_plot.max(), 1e-12)

        draw_weighted_graph(axes[row_index, 0], W_true_plot, positions, f"target: {row['graph']}", "0.25", 1e-12, width_scale)
        draw_weighted_graph(axes[row_index, 1], W_hat_plot, positions, f"learned: {row['filter']}", "0.25", 1e-12, width_scale)
        draw_difference_graph(axes[row_index, 2], W_diff, positions, "abs(W - W_hat)")

    plt.tight_layout()


plot_graph_comparisons(single, single_cases)


## Experimentos repetidos


In [ ]:
records = []
for repetition in range(REPETITIONS):
    for row in run_once(SEED + 1000 + repetition)[0]:
        records.append({"repetition": repetition, **row})
    print(f"{repetition + 1}/{REPETITIONS}")


## Estatísticas

Para cada métrica e para cada combinação grafo/filtro, reportamos média, mínimo, mediana e máximo.


In [ ]:
statistics_df = pd.DataFrame(summarize(records))
statistics_df.to_csv("statistics.csv", index=False)
statistics_df


## Gráficos das estatísticas

A figura abaixo usa o `plot` do pandas para comparar a média de cada métrica nos quatro casos.


In [ ]:
statistics_mean = statistics_df.pivot_table(
    index=["graph", "filter"],
    columns="metric",
    values="mean",
)

axes = statistics_mean.plot(
    kind="bar",
    subplots=True,
    layout=(4, 2),
    figsize=(14, 12),
    legend=False,
    title=list(statistics_mean.columns),
)
plt.tight_layout()
